# Finalisasi Kandidat Kantor BUMN/Pemerintah

**Tujuan:** Menggabungkan seluruh hasil verifikasi manual (Google Maps) ke
dalam satu daftar final kandidat Kantor, menggantikan hasil geocoding
otomatis sebelumnya yang terbukti kurang akurat untuk sebagian titik.

**Perubahan dari notebook sebelumnya:**
1. 38 kantor BUMN memakai koordinat hasil verifikasi manual (bukan
   hasil geocoding otomatis lagi)
2. **PT Bhanda Ghara Reksa** dan **PT Telkom Bandung** dihapus dari daftar
   (terkonfirmasi tutup/tidak beroperasi di alamat tersebut)
3. Ditambahkan **15 kantor Pos Indonesia** baru dari data resmi (termasuk
   1 yang menggantikan entri lama "PT Pos Indonesia" dengan koordinat
   yang lebih akurat)
4. Semua koordinat di-snap ulang ke jaringan jalan dan difilter ulang
   sesuai batas Kota Bandung

**Data lama (`candidates_J.csv`, Himpunan I) tetap tidak disentuh.**

## 1. Import library

In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
UTM_EPSG = 32748
KLASTER_NAME = "C_KantorBUMN"

## 2. Daftar final kandidat kantor (koordinat terverifikasi manual)

Seluruh koordinat di bawah ini sudah dicek langsung lewat Google Maps,
bukan hasil geocoding otomatis.

In [ ]:
kantor_final = {
    # -- Sudah diverifikasi manual (batch 1: yang tadinya bentrok koordinat) --
    "PT PLN UID Jawa Barat": (-6.920844283849016, 107.60841493895978),
    "PT Asuransi Jiwasraya Kanwil Bandung": (-6.920757233162182, 107.60685726779576),
    "PT Bank Rakyat Indonesia (BRI)": (-6.920887842754969, 107.60737166377868),
    "Bank Mandiri KC Asia Afrika": (-6.921455185329501, 107.61342899663163),
    "PT Biofarma": (-6.899251825915911, 107.60044371197489),
    "RSUP Dr. Hasan Sadikin (RSHS)": (-6.897736848378352, 107.59838886964661),
    "Stasiun Bandung (KAI)": (-6.914553632668485, 107.60200855430335),
    "Perum DAMRI": (-6.912466076137372, 107.60285399478016),
    "PT Jasa Raharja": (-6.938674670192259, 107.65909297431767),
    "Perum Bulog": (-6.938303085693414, 107.66266179663168),
    "Perum Perhutani": (-6.93706137919921, 107.70088569663167),
    "PD Kebersihan": (-6.8982770720919255, 107.63346868313907),
    "Perum Pembangunan Perumahan Nasional (Perumnas)": (-6.897732594808576, 107.63187797994912),
    "PDAM Tirtawening Kota Bandung": (-6.8958451223297175, 107.61043770827254),
    "PT ASABRI": (-6.905138523853769, 107.62187611012389),
    "PT Adhi Karya": (-6.9041856193217415, 107.62477156779553),
    "PT Angkasa Pura II": (-6.903485214858979, 107.57924116569545),
    "PT Asuransi Jasa Indonesia (Jasindo)": (-6.909341323816985, 107.60878314081096),
    "PT Bank Negara Indonesia (BNI)": (-6.913662694732189, 107.60749346964687),
    "PT Bank Tabungan Negara (BTN)": (-6.913641054291736, 107.61360885383752),
    "PT Barata Indonesia": (-6.895405582749093, 107.6433953711641),
    "PT Dirgantara Indonesia (IPTN)": (-6.898085657999985, 107.58249805615439),
    "PT INTI": (-6.937859738395585, 107.60725148313925),
    "PT Jasa Marga": (-6.88384198371409, 107.5735702369154),
    "PT Kereta Api Indonesia": (-6.913704324243904, 107.6061990236163),
    "PT Kimia Farma": (-6.908203521398592, 107.6040849101239),
    "PT Len Industri": (-6.949371442838622, 107.6194399677959),
    "PT PPRO BIJB Aerocity Development": (-6.946160091665756, 107.63954159663173),
    "PT Permodalan Nasional Madani": (-6.906419218515496, 107.64681145430328),
    "PT Pertamina Bandung": (-6.936712622811805, 107.6944990798813),
    "PT Pertani": (-6.923320121959477, 107.61907214135016),
    "PT Perusahaan Gas Negara Rayon Bandung": (-6.91901933117914, 107.64015788128793),
    "PT Pindad": (-6.936094087406148, 107.64717306964701),
    "PT Sarinah": (-6.9197399968824245, 107.60922837292446),
    "PT Sucofindo": (-6.943031091159927, 107.58518241012415),
    "PT Waskita Karya": (-6.921399726942857, 107.61399126779561),
    "Perum Pegadaian": (-6.9280014841219595, 107.60924645245214),

    # -- Kantor Pos Indonesia (data resmi, menggantikan "PT Pos Indonesia" lama) --
    "Kantor Pos Bandung (Asia Afrika)": (-6.920675732139451, 107.60621979565971),
    "Kantor Pusat Pos Cilaki": (-6.901977177542973, 107.61979108068205),
    "Kantor Pusat Pos Banda": (-6.906712259379468, 107.61743739239084),
    "SPP Pos Bandung": (-6.9432339135263685, 107.65087239228664),
    "Pos Logistik Bandung": (-6.914849439670341, 107.63485147116397),
    "Pos LE Imigrasi Bandung": (-6.899007, 107.631594),
    "Pos Bandung Babakan Ciparay": (-6.926633, 107.578454),
    "Pos Bandung Sumbersari Indah": (-6.934058, 107.573047),
    "Pos LE Mall Pelayanan Publik": (-6.915935, 107.633344),
    "Pos Bandung Turangga": (-6.935164, 107.637254),
    "Pos Bandung Ciwastra": (-6.957874, 107.657319),
    "Pos Bandung Antapani": (-6.915841, 107.656957),
    "Pos Ujungberung": (-6.913103, 107.694531),
    "Pos Ujungberung Alun-Alun": (-6.913186, 107.701846),
    "Pos Ujungberung Arcamanik": (-6.938250, 107.676936),
}

print(f"Total kandidat kantor final: {len(kantor_final)}")
print("(PT Bhanda Ghara Reksa dan PT Telkom Bandung sengaja tidak dimasukkan -- tutup/tidak beroperasi)")

Total kandidat kantor final: 52
(PT Bhanda Ghara Reksa dan PT Telkom Bandung sengaja tidak dimasukkan -- tutup/tidak beroperasi)


## 3. Ubah jadi GeoDataFrame

In [3]:
df = pd.DataFrame(
    [(nama, lat, lon) for nama, (lat, lon) in kantor_final.items()],
    columns=["nama", "lat", "lon"]
)
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326"
).to_crs(f"EPSG:{UTM_EPSG}")
gdf.head()

,nama,lat,lon,geometry
0,PT PLN UID Jawa Barat,-6.920844,107.608415,POINT (788247.729 9234210.474)
1,PT Asuransi Jiwasraya Kanwil Bandung,-6.920757,107.606857,POINT (788075.533 9234221.052)
2,PT Bank Rakyat Indonesia (BRI),-6.920888,107.607372,POINT (788132.336 9234206.287)
3,Bank Mandiri KC Asia Afrika,-6.921455,107.613429,POINT (788801.821 9234139.825)
4,PT Biofarma,-6.899252,107.600444,POINT (787379.318 9236604.673)


## 4. Muat graf jaringan jalan & snap ke node terdekat

In [4]:
G = ox.load_graphml(PROCESSED_DIR / "bandung_drive_utm48s.graphml")

nearest_nodes, dists = ox.distance.nearest_nodes(
    G, gdf.geometry.x, gdf.geometry.y, return_dist=True
)
gdf["nearest_node"] = nearest_nodes
gdf["dist_to_node_m"] = dists
print(f"{len(gdf)} titik berhasil di-snap ke jaringan jalan")

52 titik berhasil di-snap ke jaringan jalan


## 5. Filter wilayah -- buang yang di luar Kota Bandung

In [5]:
kelurahan_cache = RAW_DIR / "kelurahan_bandung.json"

if kelurahan_cache.exists():
    kelurahan = gpd.read_file(kelurahan_cache).set_crs("EPSG:4326", allow_override=True)
    boundary = kelurahan.to_crs(f"EPSG:{UTM_EPSG}").geometry.unary_union

    before = len(gdf)
    di_dalam = gdf.geometry.within(boundary)
    di_luar = gdf[~di_dalam]

    if len(di_luar) > 0:
        print(f"{len(di_luar)} dibuang (di luar Kota Bandung):")
        print(di_luar["nama"].tolist())

    gdf = gdf[di_dalam].reset_index(drop=True)
    print(f"\nSisa: {len(gdf)} dari {before}")
else:
    print("File batas kelurahan tidak ditemukan -- filter wilayah dilewati!")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_46688\3930368282.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundary = kelurahan.to_crs(f"EPSG:{UTM_EPSG}").geometry.unary_union



Sisa: 52 dari 52


## 6. Cek ulang -- pastikan tidak ada koordinat bentrok persis

In [6]:
gdf_wgs84 = gdf.to_crs("EPSG:4326")
cek = gdf_wgs84.assign(lon=gdf_wgs84.geometry.x, lat=gdf_wgs84.geometry.y)
bentrok = cek.duplicated(subset=["lat", "lon"]).sum()
print(f"Titik bentrok koordinat persis: {bentrok}")
assert bentrok == 0, "Masih ada yang bentrok, cek data manual lagi"
print("Aman, tidak ada yang bentrok.")

Titik bentrok koordinat persis: 0
Aman, tidak ada yang bentrok.


## 7. Simpan hasil final

In [8]:
gdf_wgs84["klaster"] = KLASTER_NAME
out = gdf_wgs84[["nama", "klaster", "nearest_node", "dist_to_node_m"]].copy()
out["lon"] = gdf_wgs84.geometry.x
out["lat"] = gdf_wgs84.geometry.y

out_path = PROCESSED_DIR / "candidates_kantor_bumn.csv"
out.to_csv(out_path, index=False)
print(f"Selesai. {len(out)} kandidat kantor final tersimpan di: {out_path}")

Selesai. 52 kandidat kantor final tersimpan di: d:\Magang\Week 1\spklu_bandung\data\processed\candidates_kantor_bumn.csv


## 8. Gabung dengan Himpunan J (SPBU + Mall) yang sudah ada

In [9]:
candidates_lama = pd.read_csv(PROCESSED_DIR / "candidates_J.csv")
gabungan = pd.concat([candidates_lama, out], ignore_index=True)

gabungan.to_csv(PROCESSED_DIR / "candidates_J_plus_bumn.csv", index=False)
print(f"Total kandidat gabungan: {len(gabungan)}")
gabungan["klaster"].value_counts()

Total kandidat gabungan: 142


klaster
A_SPBU          65
C_KantorBUMN    52
B_Mall          25
Name: count, dtype: int64